In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
base_path = Path(r"data/concate_data")
green_path = base_path / "green_tripdata_2025_all.parquet"
yellow_path = base_path / "yellow_tripdata_2025_all.parquet"

green_df = pd.read_parquet(green_path)
green_df = green_df.rename(columns={"lpep_pickup_datetime": "pickup_datetime", "lpep_dropoff_datetime": "dropoff_datetime"})
yellow_df = pd.read_parquet(yellow_path)
yellow_df = yellow_df.rename(columns={"tpep_pickup_datetime": "pickup_datetime", "tpep_dropoff_datetime": "dropoff_datetime"})

print(f"Green shape: {green_df.shape}")
print(f"Yellow shape: {yellow_df.shape}")

Green shape: (543139, 21)
Yellow shape: (44417596, 20)


In [3]:
yellow_df.head()

,VendorID,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [4]:
green_df.head()

,VendorID,pickup_datetime,dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-01-01 00:03:01,2025-01-01 00:17:12,N,1.0,75,235,1.0,5.93,24.70,...,0.5,6.80,0.00,NaN,1.0,34.00,1.0,1.0,0.00,0.0
1,2,2025-01-01 00:19:59,2025-01-01 00:25:52,N,1.0,166,75,1.0,1.32,8.60,...,0.5,0.00,0.00,NaN,1.0,11.10,2.0,1.0,0.00,0.0
2,2,2025-01-01 00:05:29,2025-01-01 00:07:21,N,5.0,171,73,1.0,0.41,25.55,...,0.0,0.00,0.00,NaN,1.0,26.55,2.0,2.0,0.00,0.0
3,2,2025-01-01 00:52:24,2025-01-01 01:07:52,N,1.0,74,223,1.0,4.12,21.20,...,0.5,6.13,6.94,NaN,1.0,36.77,1.0,1.0,0.00,0.0
4,2,2025-01-01 00:25:05,2025-01-01 01:01:10,N,1.0,66,158,1.0,4.71,33.80,...,0.5,7.81,0.00,NaN,1.0,46.86,1.0,1.0,2.75,0.0


In [5]:
yellow_df = yellow_df[['fare_amount','PULocationID','DOLocationID','trip_distance', 'pickup_datetime', 'dropoff_datetime']]
green_df = green_df[['fare_amount','PULocationID','DOLocationID','trip_distance', 'pickup_datetime', 'dropoff_datetime']]
taxi_df = pd.concat([yellow_df, green_df], ignore_index=True)
taxi_df = taxi_df.astype({'PULocationID': 'category', 'DOLocationID': 'category'})
taxi_df.head()

,fare_amount,PULocationID,DOLocationID,trip_distance,pickup_datetime,dropoff_datetime
0,10.0,229,237,1.60,2025-01-01 00:18:38,2025-01-01 00:26:59
1,5.1,236,237,0.50,2025-01-01 00:32:40,2025-01-01 00:35:13
2,5.1,141,141,0.60,2025-01-01 00:44:04,2025-01-01 00:46:01
3,7.2,244,244,0.52,2025-01-01 00:14:27,2025-01-01 00:20:01
4,5.8,244,116,0.66,2025-01-01 00:21:34,2025-01-01 00:25:06


In [6]:
taxi_df.dtypes

fare_amount                float64
PULocationID              category
DOLocationID              category
trip_distance              float64
pickup_datetime     datetime64[us]
dropoff_datetime    datetime64[us]
dtype: object

In [8]:
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error

taxi_df["duration"] = (
    (taxi_df["dropoff_datetime"] - taxi_df["pickup_datetime"]).dt.total_seconds() / 60
).round(2)

print(taxi_df["duration"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(2))
taxi_df = taxi_df[(taxi_df["duration"] > 0) & (taxi_df["duration"] <= 70)].copy()
taxi_df.head()

c:\Users\Martin\anaconda3\envs\autogluon_win\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


count    44960735.00
mean           17.25
std            29.79
min        -51472.32
10%             4.83
25%             8.08
50%            13.38
75%            21.25
90%            32.32
95%            42.90
99%            69.93
max         14880.77
Name: duration, dtype: float64


,fare_amount,PULocationID,DOLocationID,trip_distance,pickup_datetime,dropoff_datetime,duration
0,10.0,229,237,1.60,2025-01-01 00:18:38,2025-01-01 00:26:59,8.35
1,5.1,236,237,0.50,2025-01-01 00:32:40,2025-01-01 00:35:13,2.55
2,5.1,141,141,0.60,2025-01-01 00:44:04,2025-01-01 00:46:01,1.95
3,7.2,244,244,0.52,2025-01-01 00:14:27,2025-01-01 00:20:01,5.57
4,5.8,244,116,0.66,2025-01-01 00:21:34,2025-01-01 00:25:06,3.53


In [9]:
# Restrict model inputs to only these features
feature_cols = ["fare_amount", "PULocationID", "DOLocationID", "trip_distance", "duration"]
fare_cols = [col for col in feature_cols if col != "duration"]
duration_cols = [col for col in feature_cols if col != "fare_amount"]
taxi_df = taxi_df[feature_cols].dropna().copy()
print(fare_cols)
print(duration_cols)

['fare_amount', 'PULocationID', 'DOLocationID', 'trip_distance']
['PULocationID', 'DOLocationID', 'trip_distance', 'duration']


In [11]:
# Train/test split (model fit will use train only)
train_df, test_df = train_test_split(taxi_df, test_size=0.2, random_state=42)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (35218526, 5)
Test shape: (8804632, 5)


In [30]:
from pathlib import Path

from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42
MAX_TRAIN_ROWS = 100_000
MAX_EVAL_ROWS = 20_000

model_specs = {
    "fare_amount": {
        "label": "fare_amount",
        "columns": fare_cols,
        "path": "AutogluonModels/taxi_fare_quick",
    },
    "duration": {
        "label": "duration",
        "columns": duration_cols,
        "path": "AutogluonModels/taxi_duration_quick",
    },
}

ensemble_hyperparameters = {
    "GBM": {},
    "CAT": {},
}


def prepare_model_frame(frame, columns, max_rows):
    model_frame = frame[columns].dropna().copy()
    if len(model_frame) > max_rows:
        model_frame = model_frame.sample(n=max_rows, random_state=RANDOM_STATE).reset_index(drop=True)
    return model_frame


def fit_small_deployment_model(train_frame, test_frame, spec):
    train_data = prepare_model_frame(train_frame, spec["columns"], MAX_TRAIN_ROWS)
    test_data = prepare_model_frame(test_frame, spec["columns"], MAX_EVAL_ROWS)

    print(f"{spec['label']} training rows: {len(train_data):,}")
    print(f"{spec['label']} evaluation rows: {len(test_data):,}")

    predictor = TabularPredictor(
        label=spec["label"],
        path=spec["path"],
        problem_type="regression",
        eval_metric="mean_absolute_error",
    )
    predictor.fit(
        train_data=train_data,
        presets=["medium_quality_faster_train", "optimize_for_deployment"],
        hyperparameters=ensemble_hyperparameters,
        auto_stack=False,
        num_bag_folds=0,
        num_stack_levels=0,
        # time_limit=300,
        verbosity=2,
    )
    return predictor, test_data


model_results = {}
for target_name, spec in model_specs.items():
    predictor, holdout = fit_small_deployment_model(train_df, test_df, spec)
    model_results[target_name] = {"predictor": predictor, "holdout": holdout}

Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)


fare_amount training rows: 100,000
fare_amount evaluation rows: 20,000


	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.20
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    Can't import torch
CUDA Version:       Can't get cuda version from torch
Memory Avail:       3.74 GB / 31.75 GB (11.8%)
Disk Space Avail:   122.24 GB / 441.83 GB (27.7%)
Presets specified: ['medium_quality_faster_train', 'optimize_for_deployment']
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon will save models to "c:\Users\Martin\Desktop\Gatech\6242\Project\CSE6242_Team82\AutogluonModels\taxi_fare_quick"
Train D

duration training rows: 100,000
duration evaluation rows: 20,000


	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`


(raylet) The node with node id: 4790d184fc0cb952573462d681a83c919bf7d50ed4af89004988e7bf and address: 127.0.0.1 and node name: 127.0.0.1 has been marked dead because the detector has missed too many heartbeats from it. This can happen when a 	(1) raylet crashes unexpectedly (OOM, etc.) 
	(2) raylet has lagging heartbeats due to slow network or busy workload.


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.20
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    Can't import torch
CUDA Version:       Can't get cuda version from torch
Memory Avail:       3.85 GB / 31.75 GB (12.1%)
Disk Space Avail:   116.27 GB / 441.83 GB (26.3%)
Presets specified: ['medium_quality_faster_train', 'optimize_for_deployment']
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon will save models to "c:\Users\Martin\Desktop\Gatech\6242\Project\CSE6242_Team82\AutogluonModels\taxi_duration_quick"
Train Data Rows:    100000
Train Data Columns: 3
Label Column:       duration
Problem Type:       regression
Preprocessing data ...
Using Feature Genera

[1000]	valid_set's l1: 3.94532
[2000]	valid_set's l1: 3.93798


	-3.9338	 = Validation score   (-mean_absolute_error)
	12.99s	 = Training   runtime
	0.04s	 = Validation runtime
Fitting model: CatBoost ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting with cpus=10, gpus=0
	-3.8805	 = Validation score   (-mean_absolute_error)
	649.91s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting 1 model on all data | Fitting with cpus=16, gpus=0, mem=0.0/2.7 GB
	Ensemble Weights: {'CatBoost': 0.588, 'LightGBM': 0.412}
	-3.8177	 = Validation score   (-mean_absolute_error)
	0.05s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 673.85s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 39562.9 ro

In [31]:
for target_name, result in model_results.items():
    predictor = result["predictor"]
    predictor.save_space(remove_data=True, remove_fit_stack=True)
    print(f"Compressed {target_name} model at: {Path(predictor.path)}")

Compressed fare_amount model at: c:\Users\Martin\Desktop\Gatech\6242\Project\CSE6242_Team82\AutogluonModels\taxi_fare_quick
Compressed duration model at: c:\Users\Martin\Desktop\Gatech\6242\Project\CSE6242_Team82\AutogluonModels\taxi_duration_quick


In [32]:
evaluation_summary = {}

for target_name, spec in model_specs.items():
    predictor = model_results[target_name]["predictor"]
    holdout = model_results[target_name]["holdout"]
    label = spec["label"]

    test_pred = predictor.predict(holdout.drop(columns=[label]))
    mae = mean_absolute_error(holdout[label], test_pred)
    evaluation_summary[target_name] = mae
    print(f"{label} holdout MAE: {mae:.4f}")
    print(f"{label} leaderboard:")
    print(predictor.leaderboard(holdout, silent=True))
    print()

print("Evaluation summary:")
print(evaluation_summary)

fare_amount holdout MAE: 4.8867
fare_amount leaderboard:
                 model  score_test  score_val          eval_metric  \
0             CatBoost   -4.886701  -4.486006  mean_absolute_error   
1  WeightedEnsemble_L2   -4.886701  -4.486006  mean_absolute_error   

   pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  \
0        0.077335       0.012520  219.935704                 0.077335   
1        0.081329       0.016001  219.964031                 0.003994   

   pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  \
0                0.012520         219.935704            1       True   
1                0.003481           0.028327            2       True   

   fit_order  
0          1  
1          2  

duration holdout MAE: 4.0271
duration leaderboard:
                 model  score_test  score_val          eval_metric  \
0  WeightedEnsemble_L2   -4.027070  -3.817700  mean_absolute_error   
1             CatBoost   -4.067261  -3.880450  mean_absolu